## Example 1: Logfire Configuration and Basic Structured Logging

In [ ]:
# Example 1: Logfire Configuration and Basic Structured Logging
print("=" * 70)
print(f"🚀 Example 1: Logfire Configuration and Structured Logging")
print("=" * 70)

import logfire
from logfire.testing import TestExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

exporter = TestExporter()
logfire.configure(service_name="demo_app", send_to_logfire=False, console=False,
                  additional_span_processors=[SimpleSpanProcessor(exporter)])

# Basic structured logging with rich Python objects
user_data = {"user_id": 12345, "action": "login", "ip": "192.168.1.100"}
print(f"Logging user data: {user_data}")
logfire.info("User login attempt {action}", **user_data)
logfire.debug("Debug info", details={"request_id": "req_001"})
logfire.warn("Rate limit approaching", current_requests=95)
logfire.error("Database connection failed", error_code=500)

# Rich Python objects example
complex_data = {
    "user": {"id": 123, "name": "John Doe"},
    "metrics": {"response_time": 0.45, "requests_count": 1250},
    "metadata": {"version": "2.1.0", "environment": "production"}
}
logfire.info("Complex data structure", **complex_data)
print(f"Complex data structure: {complex_data}")

records = {span["name"]: span["attributes"] for span in exporter.exported_spans_as_dict()}
login = records["User login attempt {action}"]
assert login["logfire.msg"] == "User login attempt login" and login["user_id"] == 12345, login
assert records["Rate limit approaching"]["current_requests"] == 95, records
assert records["Database connection failed"]["error_code"] == 500, records
assert records["Debug info"]["logfire.level_num"] < records["Database connection failed"]["logfire.level_num"], records
assert "John Doe" in str(records["Complex data structure"]["user"]), records
print(f"Captured {len(records)} structured log records locally: {sorted(records)}")

print("✅ Logfire configured and logging examples completed")
print("Features demonstrated:")
print("  • Service configuration with metadata")
print("  • Structured logging with keyword arguments")
print("  • Multiple log levels (info, debug, warn, error)")
print("  • Rich Python object logging")

## Example 2: Pydantic Integration and Model Validation Logging

This example also needs `pydantic` (v2), which is not a dependency of `logfire` itself.

In [ ]:
# Example 2: Pydantic Integration and Model Validation Logging
print("=" * 70)
print(f"🚀 Example 2: Pydantic Integration and Model Validation")
print("=" * 70)

import logfire
from logfire.testing import TestExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import Optional
from datetime import datetime

exporter = TestExporter()
logfire.configure(service_name="pydantic_demo", send_to_logfire=False, console=False,
                  additional_span_processors=[SimpleSpanProcessor(exporter)])

# Pydantic model whose validator logs through Logfire
class UserModel(BaseModel):
    user_id: int
    email: str
    age: Optional[int] = None
    created_at: datetime = Field(default_factory=datetime.now)

    @field_validator('email')
    @classmethod
    def validate_email(cls, v):
        if '@' not in v:
            logfire.warn("Invalid email format detected", email=v)
            raise ValueError('Invalid email format')
        logfire.info("Email validation successful", email=v)
        return v

# Valid data
valid_data = {"user_id": 123, "email": "user@example.com", "age": 25}
logfire.info("Starting user validation", **valid_data)
user = UserModel(**valid_data)
logfire.info("User model created successfully", model_data=user.model_dump(mode="json"))

# Invalid data triggers the validator's warning and a ValidationError
invalid_data = {"user_id": 456, "email": "invalid-email", "age": -5}
logfire.info("Testing invalid data validation", **invalid_data)
try:
    UserModel(**invalid_data)
except ValidationError as e:
    assert [err["loc"] for err in e.errors()] == [("email",)], e.errors()
    logfire.error("Pydantic validation failed",
                  errors=[err["msg"] for err in e.errors()],
                  invalid_data=invalid_data)
else:
    raise AssertionError("UserModel accepted an email without '@'")

spans = exporter.exported_spans_as_dict()
names = [span["name"] for span in spans]
print(f"Captured spans: {names}")
assert names == ["Starting user validation", "Email validation successful", "User model created successfully",
                 "Testing invalid data validation", "Invalid email format detected", "Pydantic validation failed"], names
attributes = {span["name"]: span["attributes"] for span in spans}
assert attributes["Email validation successful"]["email"] == "user@example.com"
assert attributes["Invalid email format detected"]["email"] == "invalid-email"
assert '"user_id":123' in attributes["User model created successfully"]["model_data"].replace(" ", "")
assert "Invalid email format" in attributes["Pydantic validation failed"]["errors"]
assert attributes["Pydantic validation failed"]["logfire.level_num"] > attributes["Starting user validation"]["logfire.level_num"]

print("✅ Pydantic integration examples checked")

## Example 3: Advanced Error Tracking and Exception Handling

In [ ]:
# Example 3: Advanced Error Tracking and Exception Handling
print("=" * 70)
print(f"🚀 Example 3: Advanced Error Tracking and Exception Handling")
print("=" * 70)

import logfire
from logfire.testing import TestExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

exporter = TestExporter()
logfire.configure(service_name="error_tracking_demo", send_to_logfire=False, console=False,
                  additional_span_processors=[SimpleSpanProcessor(exporter)])

# Error tracking in different scenarios
def risky_division(a: int, b: int) -> float:
    logfire.info("Starting division operation", dividend=a, divisor=b)
    try:
        result = a / b
    except Exception as e:
        logfire.error("Division failed",
                      error_type=type(e).__name__,
                      error_message=str(e),
                      dividend=a, divisor=b)
        raise
    logfire.info("Division successful", result=result)
    return result

def risky_list_access(items: list, index: int):
    logfire.info("Accessing list item", list_length=len(items), target_index=index)
    try:
        result = items[index]
    except Exception as e:
        logfire.error("List access failed",
                      error_type=type(e).__name__,
                      error_message=str(e),
                      list_length=len(items), target_index=index)
        raise
    logfire.info("List access successful", value=result)
    return result

# Successful operation
assert risky_division(10, 2) == 5.0

# Error scenarios: each failure is logged with its context, then re-raised to the caller
try:
    risky_division(10, 0)
except ZeroDivisionError:
    logfire.info("Handled division by zero error")
else:
    raise AssertionError("risky_division(10, 0) should raise ZeroDivisionError")

try:
    risky_list_access([1, 2, 3], 10)
except IndexError:
    logfire.info("Handled list index error")
else:
    raise AssertionError("risky_list_access([1, 2, 3], 10) should raise IndexError")

spans = exporter.exported_spans_as_dict()
names = [span["name"] for span in spans]
print(f"Captured spans: {names}")
assert names == ["Starting division operation", "Division successful",
                 "Starting division operation", "Division failed", "Handled division by zero error",
                 "Accessing list item", "List access failed", "Handled list index error"], names
by_name = {span["name"]: span["attributes"] for span in spans}
assert by_name["Division successful"]["result"] == 5.0
division_failed = by_name["Division failed"]
assert (division_failed["error_type"], division_failed["divisor"]) == ("ZeroDivisionError", 0), division_failed
assert division_failed["error_message"] == "division by zero", division_failed
list_failed = by_name["List access failed"]
assert (list_failed["error_type"], list_failed["target_index"], list_failed["list_length"]) == ("IndexError", 10, 3), list_failed
assert division_failed["logfire.level_num"] > by_name["Division successful"]["logfire.level_num"]

print("✅ Error tracking examples checked")